In [2]:
!pip install chromadb sentence-transformers pypdf transformers# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

!pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("/content/ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

ERROR: Invalid requirement: 'transformers#': Expected end or semicolon (after name and no valid version specifier)
    transformers#
                ^
Loading PDF document...
Document Loaded
Total Characters: 2812

Preview:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: exit
Goodbye!


In [5]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.

# ==========================================================
# ADVANCED RAG LEGAL RESEARCH ASSISTANT
# Improvements:
# 1. Legal clause-aware chunking
# 2. Semantic reranking
# 3. Optimized legal prompt
# ==========================================================

# Install first if needed
# pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import re
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
# from transformers import pipeline # Removed: Using Google Gemini API instead
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Removed: Using Google Gemini API instead
import google.generativeai as genai
from google.colab import userdata


# ----------------------------------------------------------
# STEP 2 — Load Legal Document
# ----------------------------------------------------------

print("Loading legal document...")

reader = PdfReader("/content/data_privacy_regulation.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print("Document loaded successfully")
print("Total characters:", len(text))



# ----------------------------------------------------------
# STEP 3 — Legal Clause Aware Chunking
# ----------------------------------------------------------

def legal_chunk_text(text):

    """
    Split legal document by Articles
    instead of random character chunks
    """

    parts = re.split(r'(Article\s+\d+[:\-]?)', text)

    chunks = []
    current_chunk = ""

    for part in parts:

        if "Article" in part:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = part

        else:
            current_chunk += " " + part

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


print("\nCreating legal chunks...")

chunks = legal_chunk_text(text)

print("Total chunks:", len(chunks))
print("\nExample chunk:\n", chunks[0])



# ----------------------------------------------------------
# STEP 4 — Load Embedding Model
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")



# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")

print("Vector database ready")



# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored")



# ----------------------------------------------------------
# STEP 7 — Load Semantic Reranker
# ----------------------------------------------------------

print("\nLoading semantic reranker...")

rerranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker ready")



# ----------------------------------------------------------
# STEP 8 — Retriever with Reranking
# ----------------------------------------------------------

def retrieve(query, k=5):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    # Prepare pairs for reranking
    pairs = [[query, doc] for doc in docs]

    scores = reranker.predict(pairs)

    ranked_docs = [
        doc for _, doc in sorted(
            zip(scores, docs),
            reverse=True
        )
    ]

    return ranked_docs[:3]



# ----------------------------------------------------------
# STEP 9 — Load LLM (Gemini API)
# ----------------------------------------------------------

print("\nLoading LLM...")

try:
    GOOGLE_API_KEY = userdata.get('google_api_key') # Ensure your API key is stored in Colab secrets
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash')
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None



# ----------------------------------------------------------
# STEP 10 — Answer Generator
# ----------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    context_docs = retrieve(query)

    context = "\n\n".join(context_docs)

    prompt = f"""
Answer the legal question using the context.

Context:
{context}

Question:
{query}

Answer:
"""
    try:
        response = llm.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {e}"

    return answer



# ----------------------------------------------------------
# STEP 11 — Chat Loop
# ----------------------------------------------------------

print("\n====================================")
print("LEGAL RESEARCH ASSISTANT READY")
print("Type 'exit' to stop")
print("====================================\n")

while True:

    question = input("Ask a legal question: ")

    if question.lower() == "exit":
        print("Session ended")
        break

    answer = answer_question(question)

    print("\nAnswer:")
    print(answer)

    print("\n" + "-"*60 + "\n")

Loading legal document...
Document loaded successfully
Total characters: 2885

Creating legal chunks...
Total chunks: 8

Example chunk:
 Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready

Creating embeddings...
All chunks stored

Loading semantic reranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

Reranker ready

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully

LEGAL RESEARCH ASSISTANT READY
Type 'exit' to stop

Ask a legal question: What are the responsibilities of organizations handling personal data?

Answer:
Data Controller

------------------------------------------------------------

Ask a legal question: What penalties are mentioned for non-compliance?

Answer:
financial penalties, regulatory investigations, and operational restrictions

------------------------------------------------------------

Ask a legal question: exit
Session ended


### Scenario: University Library Assistant
 A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
 How It Works
- Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
 - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
 - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
- Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
- LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
 - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.


In [7]:
# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install chromadb sentence-transformers pypdf transformers

In [8]:
# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# from transformers import pipeline # Removed: Using Google Gemini API instead
import google.generativeai as genai
from google.colab import userdata

In [9]:
# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [10]:
# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("library_docs")
except:
    pass

collection = client.create_collection("library_docs")
print("Vector database ready")

Vector database ready


In [11]:
# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

try:
    GOOGLE_API_KEY = userdata.get('google_api_key') # Ensure your API key is stored in Colab secrets
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash')
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully


In [12]:
# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [13]:
# --------------------------------------------------------------
# STEP 7 — Process PDF Document
# --------------------------------------------------------------

print("Processing PDF...")

reader = PdfReader("/content/Introduction_to_Data_Science.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

chunks = chunk_text(text)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print(f"Document processed successfully. {len(chunks)} chunks stored.")

Processing PDF...
Total chunks: 8
Document processed successfully. 8 chunks stored.


In [14]:
# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs

In [15]:
# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a university library assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in the document".

Context:
{context}

Question: {query}

Provide a clear and concise answer.
"""
    try:
        response = llm.generate_content(prompt)
        result = response.text
    except Exception as e:
        result = f"Error generating answer: {e}"

    print("\nRaw Model Output:\n", result)

    return result

In [16]:
# --------------------------------------------------------------
# STEP 10 — Chat Loop
# --------------------------------------------------------------

print("\n===================================")
print("UNIVERSITY LIBRARY ASSISTANT READY")
print("Type 'exit' to stop")
print("===================================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Session ended")
        break

    answer = answer_question(question)

    print("\nAnswer:\n")
    print(answer)

    print("\n" + "-"*60 + "\n")


UNIVERSITY LIBRARY ASSISTANT READY
Type 'exit' to stop

Ask a question: how many books in liberary

Retrieved Chunks:
 ['ilding models that learn patterns from data.\n\x7f\nData Visualization – Presenting insights using charts and dashboards.\n2. The Data Science Workflow\n1\nDefine the problem or research question.\n2\nCollect relevant data from databases, APIs, or files.\n3\nClean and preprocess the data.\n4\nExplore the data using statistics and visualization.\n5\nBuild predictive or analytical models.\n6\nEvaluate and interpret results.\n7\nCommunicate insights to stakeholders.\n3. Machine Learning in Data Science\nMachine learning', 'ke data-driven decisions.\nWith the rapid growth of data, the demand for skilled data scientists continues to rise.\nUnderstanding the basics of data science, machine learning, and data analysis is an essential skill\nfor students and professionals in the modern digital economy.\n', 'achine learning library.\n\x7f\nTensorFlow / PyTorch – Deep learnin

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Raw Model Output:
 
You are a university library assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in the document".

Context:
ilding models that learn patterns from data.

Data Visualization – Presenting insights using charts and dashboards.
2. The Data Science Workflow
1
Define the problem or research question.
2
Collect relevant data from databases, APIs, or files.
3
Clean and preprocess the data.
4
Explore the data using statistics and visualization.
5
Build predictive or analytical models.
6
Evaluate and interpret results.
7
Communicate insights to stakeholders.
3. Machine Learning in Data Science
Machine learning ke data-driven decisions.
With the rapid growth of data, the demand for skilled data scientists continues to rise.
Understanding the basics of data science, machine learning, and data analysis is an essential skill
for students and professionals in the modern digital economy.
 achine learning library.

T

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Raw Model Output:
 
You are a university library assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in the document".

Context:
achine learning library.

TensorFlow / PyTorch – Deep learning frameworks.
5. Applications of Data Science

Healthcare – Disease prediction and medical imaging.

Finance – Fraud detection and risk analysis.

E-commerce – Recommendation systems.

Transportation – Traffic prediction and route optimization.

Social Media – Sentiment analysis and trend detection.
Conclusion
Data Science is transforming industries by enabling organizations to make data-driven decisions.
With the rapid growth of ke data-driven decisions.
With the rapid growth of data, the demand for skilled data scientists continues to rise.
Understanding the basics of data science, machine learning, and data analysis is an essential skill
for students and professionals in the modern digital economy.
 ilding models that learn pat

In [24]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers google-generativeai


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# Removed transformers pipeline as we are switching to Gemini API
import google.generativeai as genai
from google.colab import userdata # Used to securely store your API key


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")
print("Vector database ready")


# --------------------------------------------------------------
# STEP 5 — Load Language Model (Gemini)
# --------------------------------------------------------------

print("Loading LLM...")

# Configure Gemini API key from Colab secrets
# You need to create a secret named 'google_api_key' in Colab's Secrets tab
# If you don't have an API key, get one from https://makersuite.google.com/k/apikey
try:
    GOOGLE_API_KEY = userdata.get('google_api_key')
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a capable alternative to gemini-2
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None # Set LLM to None to prevent further errors


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in document".

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    try:
        response = llm.generate_content(prompt)
        result = response.text
    except Exception as e:
        result = f"Error generating answer: {e}"

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...
Error loading Gemini LLM: Requesting secret google_api_key timed out. Secrets can only be fetched when running from the Colab UI.
Please ensure GOOGLE_API_KEY is set in Colab secrets and you have access to gemini-2.5-flash.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1e1457bda5b76aff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers google-generativeai


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# Removed transformers pipeline as we are switching to Gemini API
import google.generativeai as genai
from google.colab import userdata # Used to securely store your API key


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")
print("Vector database ready")


# --------------------------------------------------------------
# STEP 5 — Load Language Model (Gemini)
# --------------------------------------------------------------

print("Loading LLM...")

# Configure Gemini API key from Colab secrets
# You need to create a secret named 'google_api_key' in Colab's Secrets tab
# If you don't have an API key, get one from https://makersuite.google.com/k/apikey
try:
    GOOGLE_API_KEY = userdata.get('google_api_key')
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a capable alternative to gemini-2
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None # Set LLM to None to prevent further errors


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in document".

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    try:
        response = llm.generate_content(prompt)
        result = response.text
    except Exception as e:
        result = f"Error generating answer: {e}"

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...
Error loading Gemini LLM: Requesting secret google_api_key timed out. Secrets can only be fetched when running from the Colab UI.
Please ensure GOOGLE_API_KEY is set in Colab secrets and you have access to gemini-2.5-flash.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1e1457bda5b76aff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers google-generativeai


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# Removed transformers pipeline as we are switching to Gemini API
import google.generativeai as genai
from google.colab import userdata # Used to securely store your API key


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")
print("Vector database ready")


# --------------------------------------------------------------
# STEP 5 — Load Language Model (Gemini)
# --------------------------------------------------------------

print("Loading LLM...")

# Configure Gemini API key from Colab secrets
# You need to create a secret named 'google_api_key' in Colab's Secrets tab
# If you don't have an API key, get one from https://makersuite.google.com/k/apikey
try:
    GOOGLE_API_KEY = userdata.get('google_api_key')
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a capable alternative to gemini-2
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None # Set LLM to None to prevent further errors


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in document".

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    try:
        response = llm.generate_content(prompt)
        result = response.text
    except Exception as e:
        result = f"Error generating answer: {e}"

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...
Error loading Gemini LLM: Requesting secret google_api_key timed out. Secrets can only be fetched when running from the Colab UI.
Please ensure GOOGLE_API_KEY is set in Colab secrets and you have access to gemini-2.5-flash.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1e1457bda5b76aff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers google-generativeai


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# Removed transformers pipeline as we are switching to Gemini API
import google.generativeai as genai
from google.colab import userdata # Used to securely store your API key


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")
print("Vector database ready")


# --------------------------------------------------------------
# STEP 5 — Load Language Model (Gemini)
# --------------------------------------------------------------

print("Loading LLM...")

# Configure Gemini API key from Colab secrets
# You need to create a secret named 'google_api_key' in Colab's Secrets tab
# If you don't have an API key, get one from https://makersuite.google.com/k/apikey
try:
    GOOGLE_API_KEY = userdata.get('google_api_key')
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel('gemini-2.5-flash') # Using gemini-pro as a capable alternative to gemini-2
    print("Gemini LLM loaded successfully")
except Exception as e:
    print(f"Error loading Gemini LLM: {e}")
    print("Please ensure 'google_api_key' is set in Colab secrets and you have access to gemini-2.5-flash.")
    llm = None # Set LLM to None to prevent further errors


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):
    if llm is None:
        return "LLM not initialized. Please check API key configuration."

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in document".

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    try:
        response = llm.generate_content(prompt)
        result = response.text
    except Exception as e:
        result = f"Error generating answer: {e}"

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...
Error loading Gemini LLM: Requesting secret google_api_key timed out. Secrets can only be fetched when running from the Colab UI.
Please ensure GOOGLE_API_KEY is set in Colab secrets and you have access to gemini-2.5-flash.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1e1457bda5b76aff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [34]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + FLAN-T5
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")

print("Vector database ready")


# --------------------------------------------------------------
# STEP 5 — Load FLAN-T5 Model
# --------------------------------------------------------------

print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Chunking Function
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    if not docs:
        return "No relevant information found."

    context = " ".join(docs)

    prompt = f"""
Answer the legal question using the context.

Context:
{context}

Question:
{query}

Answer clearly:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(label="Ask a Legal Question")

    answer_box = gr.Textbox(label="Answer", lines=10)

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e21e5371e16bab647.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scenario: Healthcare Clinical Research Assistant
Context

Hospitals and medical research teams regularly analyze large medical documents such as:

clinical trial reports

treatment guidelines

drug safety reports

patient care protocols

These documents are very long and technical, often containing:

medical terminology

treatment procedures

dosage guidelines

adverse drug reactions

Doctors and healthcare researchers often spend hours searching for specific information, such as:

recommended drug dosages

treatment protocols for specific diseases

safety warnings and contraindications

In [ ]:
# Scenario: "The Healthcare Policy Navigator"
# Background
# You are part of the Healthcare Compliance & Policy Team at a large hospital network.
# New government regulations on patient data privacy and telemedicine practices have just been released.
# The hospital must quickly adapt to ensure compliance and avoid penalties.
# Challenge
# The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app). Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
# - Generate clear, actionable answers that can guide doctors, administrators, and IT staff

In [31]:
# ==============================================================
# RAG HEALTHCARE POLICY NAVIGATOR
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# STEP 1 — Install Dependencies
!pip install gradio chromadb sentence-transformers pypdf transformers


# STEP 2 — Import Libraries
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
# Removed google.generativeai and userdata as we are using Hugging Face pipeline
from transformers import pipeline


# STEP 3 — Load Embedding Model
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

# STEP 4 — Initialize Vector Database (Chroma)
client = chromadb.Client()

try:
    client.delete_collection("healthcare_policies")
except:
    pass

collection = client.create_collection("healthcare_policies")
print("Vector database ready")


# STEP 5 — Load Language Model (Flan-T5)
print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM ready")


# STEP 6 — Document Chunking
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# STEP 7 — Process PDF Document
def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# STEP 8 — Retriever
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# STEP 9 — Answer Generation
def answer_question(query):
    # Note: When using a Hugging Face pipeline, there's no direct `llm is None` check
    # if the pipeline was initialized successfully. The error handling for LLM
    # will be different than with the Gemini API.

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a healthcare policy assistant. Your goal is to provide concise and accurate answers based on the provided healthcare policy document.

Use ONLY the context below to answer the question. If the answer is not in the context, say "Not found in the document".

Context:
{context}

Question: {query}

Provide a clear and concise answer.
"""

    # Using the Hugging Face pipeline for text generation
    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )
    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# STEP 10 — Chat Function
def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# STEP 11 — Build Gradio Interface
with gr.Blocks() as demo:

    gr.Markdown("# ⚕️ Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation document and ask questions about:

• patient data privacy
• telemedicine practices
• compliance requirements
• penalties for violations
""")

    pdf_input = gr.File(label="Upload Healthcare Policy PDF", value="/content/National Telemedicine Privacy Act, 2026.pdf", interactive=False)

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Healthcare Policy Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# STEP 12 — Launch Application
demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM ready
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41160158bcc5dd4c5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
# ==========================================================
# HEALTHCARE POLICY NAVIGATOR
# GRADIO + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# STEP 1 — Load Language Model
print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM ready")


# STEP 2 — Load Embedding Model
print("Loading embeddings...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")


# STEP 3 — Initialize Vector Database
client = chromadb.Client()

try:
    client.delete_collection("policy_docs")
except:
    pass

collection = client.create_collection("policy_docs")


# ----------------------------------------------------------
# STEP 4 — Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# STEP 5 — Process Uploaded Policy PDF

def process_pdf(file):
    reader = PdfReader(file.name)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Policy document processed successfully. {len(chunks)} sections indexed."


# STEP 6 — Retriever
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 7 — Generate Compliance Answer
def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a Healthcare Policy Navigator.

Use the policy context below to answer the question.

Provide a clear and actionable answer that hospital staff
(doctors, administrators, or IT teams) can follow.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm(
        prompt,
        max_length=220,
        temperature=0.2
    )

    return response[0]["generated_text"]


# STEP 8 — Chat Function
def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)


# STEP 9 — Gradio Interface
with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation document and ask questions about:

• Patient data privacy
• Telemedicine rules
• Compliance requirements
• Penalties for violations
""")

    pdf_input = gr.File(label="Upload Healthcare Regulation PDF")
    upload_button = gr.Button("Process Policy Document")
    status = gr.Textbox(label="System Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Policy Guidance",
        lines=12
    )

    ask_button = gr.Button("Get Guidance")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )

# STEP 10 — Launch App
demo.launch()

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM ready
Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8386d9bf67c87b0e5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scenario: "The Environmental Policy Compliance Assistant"
Background
You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
The company must ensure compliance to avoid fines and reputational damage.

Challenge
The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
Your task is to:
- Process the regulation document so the assistant can store and understand it.
- Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
- Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

Roles
- Learner (You): Environmental compliance officer using the assistant.
- Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
- Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

🔄 Flow of the Scenario
- Upload Environmental Regulation PDF
Example: “National Carbon Emissions Act 2026”.
- System Processes Document
- Splits into chunks.
- Embeds into vector database.
- Stores for retrieval.
- Ask Questions
- “What is the maximum carbon emission allowed per factory per year?”
- “What penalties apply if hazardous waste is not disposed of properly?”
- “What renewable energy targets must we meet by 2030?”
- Assistant Responds
- Retrieves relevant chunks.
- Generates compliance-focused answers.
- Provides short, clear guidance.
- Outcome
- Learners practice extracting environmental obligations.
- Managers receive summarized compliance insights.
- Executives gain confidence in sustainability strategy alignment.

🎯 Training Objective
This scenario helps learners:
- Understand how RAG systems can support environmental compliance.
- Practice formulating precise queries to extract obligations.
- Experience how AI can simplify complex environmental regulations into actionable steps.

👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

In [35]:
# ==============================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT (RAG)
# PDF + EMBEDDINGS + CHROMA + FLAN-T5 + GRADIO
# ==============================================================
# STEP 1 — Install Dependencies
!pip install gradio chromadb sentence-transformers pypdf transformers


# STEP 2 — Import Libraries
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM



# STEP 3 — Load Embedding Model
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# STEP 4 — Initialize Vector Database
client = chromadb.Client()

try:
    client.delete_collection("environmental_docs")
except:
    pass

collection = client.create_collection("environmental_docs")

print("Vector database ready")


# STEP 5 — Load FLAN-T5 Model
print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("LLM loaded successfully")


# STEP 6 — Chunking Function
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# STEP 7 — Process Environmental Regulation PDF
def process_pdf():

    file_path = "/content/National Carbon Emissions & Sustainability Act, 2026.pdf"

    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks created:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed successfully. {len(chunks)} chunks stored."



# STEP 8 — Retriever
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# STEP 9 — Answer Generation
def answer_question(query):

    docs = retrieve(query)

    if not docs:
        return "No relevant regulation found."

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Compliance Assistant.

Use ONLY the regulation context below to answer.

Context:
{context}

Question:
{query}

Provide a short compliance-focused answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


# STEP 10 — Chat Function
def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    return answer


# STEP 11 — Gradio Interface
with gr.Blocks() as demo:

    gr.Markdown("# 🌱 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload environmental regulation documents and ask questions about:

• Carbon emission limits
• Hazardous waste disposal
• Renewable energy targets
• Environmental penalties
""")

    process_button = gr.Button("Process Environmental Regulation")

    status_box = gr.Textbox(label="Processing Status")

    process_button.click(
        process_pdf,
        outputs=status_box
    )

    question_box = gr.Textbox(
        label="Ask Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Assistant Response",
        lines=10
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )

# STEP 12 — Launch App
demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36b80aed60b031c7b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
